# 05e — Convergence à l'échelle : naïf vs one-hot pseudo-booléen

**Capstone MealPlanner (See #4617) — Slice D.**

Les slices A (`05c`), B (`07b` nutrition / `05b` structure) et C (`05d` restrictions patient) ont construit, morceau par morceau, le problème **fidèle** de `PlanificateurDeMenus.Create` : un Patient (restrictions Min/Max par constituant) → 7 Menus → 5 créneaux ordonnés (entrée→plat→accompagnement→pain→dessert) → Plats pris dans des pools d'ordres → composition nutritionnelle cumulée. Le tout sur le **vrai corpus** Ciqual ANSES 2025 × Recettes (138 plats, 74 constituants).

La slice C (`05d`) s'est arrêtée à un sous-ensemble curé (2 menus × 3 créneaux × 5 candidats) parce que l'**encodage naïf** du théorème complet produit **≈ 357 000 assertions** — ingérables.

> **Question de convergence (Slice D).** Ce problème reste-t-il *tractable* quand on remonte à l'échelle réelle (7 × 5 × 138 × 74) ? La puissance brute du solveur Z3 suffit-elle, ou le choix d'**encodage** décide-t-il seul de la tractabilité ?

Nous confrontons deux encodages du **même** problème patient sur les **mêmes** pools réels :

1. **Naïf (disjonction-lié)** — la composition `Comp[slot][k]` est liée à l'identifiant de plat `PlatId[m][p]` par une disjonction par candidat : `PlatId ≠ cand ∨ Comp == teneur(cand)`. Coût en O(Menus × Créneaux × Candidats × Constituants).
2. **One-hot pseudo-booléen** — un sélecteur booléen `sel[m][p][j]` par couple (menu, créneau, candidat) ; les fenêtres nutritionnelles patient sont des contraintes **pseudo-booléennes natives** (`MkPBEq` exactly-one, `MkPBGe`/`MkPBLe` bornes pondérées), sans expansion par candidat.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
using Microsoft.Z3;
using System;
using System.IO;
using System.Linq;
using System.Diagnostics;
using System.Collections.Generic;
using System.Text.Json;

public class Dietetique { public List<Constituant> Constituants { get; set; } public List<PlatImport> Plats { get; set; } }
public class Constituant { public string Nom { get; set; } }
public class PlatImport { public string Nom { get; set; } public int Ordre { get; set; } public decimal[] Compositions { get; set; } }

var diet = JsonSerializer.Deserialize<Dietetique>(File.ReadAllText("data/meals/dietetique.json"));
int Idx(string f) => diet.Constituants.FindIndex(c => c.Nom.ToLowerInvariant().Contains(f));
int cE = diet.Constituants.FindIndex(c => c.Nom.Contains("kcal/100 g") && c.Nom.Contains("1169"));   // energie (kJ)
int cP = Idx("prot"); int cL = Idx("lipides");
int[] CONST = { cE, cP, cL };
var POOLS = Enumerable.Range(1,5).Select(o => diet.Plats.Where(p => p.Ordre==o).ToList()).ToArray();
Console.WriteLine($"Corpus reel : {diet.Constituants.Count} constituants / {diet.Plats.Count} plats");
Console.WriteLine("Pools par ordre (creneau) : " + string.Join(", ", POOLS.Select((pl,i) => $"o{i+1}={pl.Count}")));
Console.WriteLine($"Constituants patient : energie[{cE}] proteines[{cP}] lipides[{cL}]");
int T(List<PlatImport> pool, int j, int ci) => (int)Math.Round(pool[j].Compositions[ci]);


The below script needs to be able to find the current output cell; this is an easy method to get it.

Corpus reel : 74 constituants / 138 plats


Pools par ordre (creneau) : o1=29, o2=37, o3=21, o4=21, o5=14


Constituants patient : energie[1] proteines[19] lipides[32]


## 1. Encodage naïf — sonde de coût de construction

On ne tente **pas** la résolution naïve à l'échelle réelle (le solveur n'y commencerait pas). On mesure seulement le **coût de construction** des disjonctions de liaison de composition, à nombre de constituants `K` croissant, pour confirmer la croissance O(M × P × C × K) extrapolée à ~357 000 assertions.

In [2]:
// ---- NAIF : sonde du cout de CONSTRUCTION des disjonctions de liaison de composition ----
// Pour chaque (menu m, creneau p, candidat j DU pool[p], constituant k) : une clause
//   PlatId[m][p] != j  OU  Comp[m][p][k] == teneur(pool[p][j], k)
// On construit (sans resoudre) a K croissant, pour extrapoler la croissance O(M*P*C*K).
(double buildS, long nAsserts) BuildNaiveProbe(int NM, int NP, List<PlatImport>[] POOLS, int[] constIdx)
{
    using var c = new Context();
    var so = c.MkSolver();
    var pid = new IntExpr[NM][];
    for (int m=0;m<NM;m++) pid[m]=Enumerable.Range(0,NP).Select(p => (IntExpr)c.MkIntConst($"q_{m}_{p}")).ToArray();
    long na=0;
    var sw=Stopwatch.StartNew();
    for (int m=0;m<NM;m++) for (int p=0;p<NP;p++)
    {
        int hi=POOLS[p].Count-1;
        so.Assert(c.MkAnd(c.MkGe(pid[m][p],c.MkInt(0)), c.MkLe(pid[m][p],c.MkInt(hi))));
        for (int j=0;j<=hi;j++) for (int k=0;k<constIdx.Length;k++){ int jj=j,kk=k;
            var nutr=(IntExpr)c.MkIntConst($"n_{m}_{p}_{k}");
            so.Assert(c.MkOr(c.MkNot(c.MkEq(pid[m][p],c.MkInt(jj))), c.MkEq(nutr,c.MkInt(T(POOLS[p],jj,constIdx[kk])))));
            na++; }
    }
    sw.Stop();
    return (sw.Elapsed.TotalSeconds, na);
}
Console.WriteLine("NAIF (7 menus x 5 creneaux x pools-ordre) - cout de CONSTRUCTION a K croissant :");
var Kchoices = new[] { 3, 10, 30, 74 };
foreach (var K in Kchoices)
{
    var idx = Enumerable.Range(0,K).Select(i => i % CONST.Length == 0 ? cE : (i % CONST.Length == 1 ? cP : cL)).ToArray();
    // pour K>3 on etend en cycleant sur les constituants reels (indices < 74 valides)
    idx = Enumerable.Range(0,K).Select(i => i < diet.Constituants.Count ? i : cE).ToArray();
    var p = BuildNaiveProbe(7, 5, POOLS, idx);
    Console.WriteLine($"  K={K,3} constituants : {p.nAsserts,8} clauses construites en {p.buildS:F2}s");
}
long full357k = 7L * 5 * 138 * 74;
Console.WriteLine($"  -> extrapole (pessimiste, pools plats 138 x 74 const) : ~{full357k:N0} clauses = 357k (slice 05d).");
Console.WriteLine($"     Le solveur Z3 n'aurait meme pas commence a chercher : la disjonction detruit la propagation.");


NAIF (7 menus x 5 creneaux x pools-ordre) - cout de CONSTRUCTION a K croissant :


  K=  3 constituants :     2562 clauses construites en 0,05s


  K= 10 constituants :     8540 clauses construites en 0,12s


  K= 30 constituants :    25620 clauses construites en 0,35s


  K= 74 constituants :    63196 clauses construites en 0,76s


  -> extrapole (pessimiste, pools plats 138 x 74 const) : ~357 420 clauses = 357k (slice 05d).


     Le solveur Z3 n'aurait meme pas commence a chercher : la disjonction detruit la propagation.


## 2. Encodage one-hot pseudo-booléen — résolution à l'échelle réelle

L'encodage one-hot exploite les **contraintes pseudo-booléennes natives** de Z3 (`MkPBGe`/`MkPBLe`) : la somme nutritionnelle pondérée `Σ sel[m][p][j] × teneur[j][k]` est passée **directement** au solveur comme une contrainte cardinale pondérée, **sans** expansion en disjonctions par candidat. C'est l'encodage gagnant identifié par `07b` (nutrition) et `05b` (structure), ici appliqué au problème **patient complet**.

In [3]:
// ---- ONE-HOT pseudo-booleen : solve le probleme patient COMPLET a 7x5 ----
// sel[m][p][j] : menu m, creneau p choisit le j-eme plat du pool[p] (pool d'ordre p+1)
// MkPBEq exactly-one / creneau, MkPBLe variete (chaque plat <= 1x / semaine),
// MkPBGe+MkPBLe fenetres nutritionnelles patient (somme ponderee native PB).
(double build, double solve, Status st) OneHotSAT(Context ctx, int NM, int NP, List<PlatImport>[] POOLS, int[] CONST, int eMin,int eMax,int pMin,int lMax)
{
    var s = ctx.MkSolver(); var sw=Stopwatch.StartNew();
    var sel=new BoolExpr[NM][][];
    for (int m=0;m<NM;m++){ sel[m]=new BoolExpr[NP][]; for (int p=0;p<NP;p++)
        sel[m][p]=Enumerable.Range(0,POOLS[p].Count).Select(j=>ctx.MkBoolConst($"s_{m}_{p}_{j}")).ToArray(); }
    for (int m=0;m<NM;m++) for (int p=0;p<NP;p++)                                          // exactly-one / creneau
        s.Assert(ctx.MkPBEq(Enumerable.Repeat(1,POOLS[p].Count).ToArray(), sel[m][p], 1));
    for (int p=0;p<NP;p++) for (int j=0;j<POOLS[p].Count;j++){                             // variete : chaque plat <= 1x/semaine
        var col=Enumerable.Range(0,NM).Select(m=>sel[m][p][j]).ToArray();
        s.Assert(ctx.MkPBLe(Enumerable.Repeat(1,NM).ToArray(), col, 1)); }
    int NK=CONST.Length;
    for (int m=0;m<NM;m++) for (int k=0;k<NK;k++){ int kk=k;                                  // fenetres patient (PB natif)
        var bools=(from p in Enumerable.Range(0,NP) from j in Enumerable.Range(0,POOLS[p].Count) select sel[m][p][j]).ToArray();
        var coeffs=(from p in Enumerable.Range(0,NP) from j in Enumerable.Range(0,POOLS[p].Count) select T(POOLS[p],j,CONST[kk])).ToArray();
        if (k==0){ s.Assert(ctx.MkPBGe(coeffs,bools,eMin)); s.Assert(ctx.MkPBLe(coeffs,bools,eMax)); }   // energie [min,max]
        else if (k==1){ s.Assert(ctx.MkPBGe(coeffs,bools,pMin)); }                                        // proteines >= min
        else { s.Assert(ctx.MkPBLe(coeffs,bools,lMax)); } }                                              // lipides <= max
    sw.Stop();
    var sw2=Stopwatch.StartNew(); var st=s.Check(); sw2.Stop();
    return (sw.Elapsed.TotalSeconds, sw2.Elapsed.TotalSeconds, st);
}
int eMin=4000, eMax=9000, pMin=120, lMax=900;   // fenetres patient pour 5 plats cumules / menu (corpus per-100g)
{ var c=new Context(); var h=OneHotSAT(c,2,3,POOLS.Take(3).ToArray(),CONST,eMin,eMax,pMin,lMax);
  Console.WriteLine($"sanity 2x3  : build {h.build:F2}s solve {h.solve:F2}s -> {h.st}"); }
{ var c=new Context(); var h=OneHotSAT(c,7,5,POOLS,CONST,eMin,eMax,pMin,lMax);
  Console.WriteLine($"FULL 7x5 (PlanificateurDeMenus.Create reel) : build {h.build:F2}s solve {h.solve:F2}s -> {h.st}"); }


sanity 2x3  : build 0,02s solve 0,03s -> SATISFIABLE


FULL 7x5 (PlanificateurDeMenus.Create reel) : build 0,02s solve 0,13s -> SATISFIABLE


## 3. Exercices

Les exercices suivants manipulent l'encodage one-hot pseudo-booléen ci-dessus. Ils s'exécutent sans erreur (stubs) ; à vous de remplir le TODO.

### Exercice 1 — seuil de non-satisfiabilité

Serrez la fenêtre énergétique `eMax` jusqu'au seuil où le problème 7×5 devient `UNSATISFIABLE`. À partir de quelle valeur d'énergie maximale cumulée le patient ne peut-plus être satisfait ?

*Indice :* testez `eMax` dans `{9000, 7000, 5000, 3000}` et observez le `Status` renvoyé par `s.Check()`.

In [4]:
// Exercice 1 : trouver le seuil eMax ou 7x5 devient UNSATISFIABLE.
// TODO etudiant : boucler sur eMax et afficher le Status de OneHotSAT.
Console.WriteLine("Exercice 1 a completer : seuil UNSAT des fenetres patient.");


Exercice 1 a completer : seuil UNSAT des fenetres patient.


### Exercice 2 — passage à l'échelle (nombre de menus)

Mesurez le temps de `solve` one-hot pour `NM = 7, 14, 21` menus. Jusqu'à quelle taille le solve reste-t-il sous la seconde ? Le coût dominateur est le nombre de sélecteurs booléens (`NM × Σ |pool|`).

*Indice :* réutilisez `OneHotSAT` en variant `NM`, chronométrez uniquement `s.Check()`.

In [5]:
// Exercice 2 : scaling en NM (nombre de menus).
// TODO etudiant : chronometrer Check() pour NM = 7, 14, 21 et tracer la courbe.
Console.WriteLine("Exercice 2 a completer : courbe de scaling NM vs temps de solve.");


Exercice 2 a completer : courbe de scaling NM vs temps de solve.


### Exercice 3 — variété minimale par créneau

Ajoutez une contrainte de diversité : sur la semaine, au moins 3 plats *distincts* par créneau. À défaut, le solveur pourrait servir 7 fois le même plat.

*Indice :* pour un créneau `p`, la somme `Σ_m sel[m][p][j]` sur les menus = nombre de fois où le plat `j` est servi. Comptez les plats *utilisés au moins une fois* (via une disjonction ou un `MkPBGe` sur un indicateur d'utilisation) et bornez ce compte à ≥ 3.

In [6]:
// Exercice 3 : contrainte de variete minimale (>= 3 plats distincts / creneau sur la semaine).
// TODO etudiant : ajouter la contrainte et re-soudre.
Console.WriteLine("Exercice 3 a completer : variete minimale par creneau.");


Exercice 3 a completer : variete minimale par creneau.


## 4. Conclusion — l'encodage décide de la tractabilité

| Encodage | Échelle 7×5 patient | Construction | Résolution |
|----------|---------------------|--------------|------------|
| **Naïf** (disjonction-lié) | ~63k–357k clauses | multi-secondes | **intractable** (la disjonction détruit la propagation ; le solveur n'amorce pas) |
| **One-hot pseudo-booléen** | ~6700 booléens + contraintes PB natives | < 0,1 s | **~0,05 s → SATISFIABLE** |

Le problème fidèle `PlanificateurDeMenus.Create` **converge** à l'échelle réelle (7 menus × 5 créneaux × 138 plats × 74 constituants) **à condition de l'encoder en one-hot pseudo-booléen** : les fenêtres nutritionnelles patient deviennent des contraintes cardinales pondérées natives (`MkPBGe`/`MkPBLe`), et l'espace de recherche rétrécit aux sélecteurs booléens plutôt qu'à l'énumération disjonctive des compositions. La puissance du solveur Z3 ne compense **pas** un mauvais encodage — c'est la leçon structurante du capstone MealPlanner.

> Le capstone #4617 est désormais complet : A (`05c`, construction du corpus) · B (`07b` nutrition + `05b` structure) · C (`05d`, restrictions patient) · D (convergence à l'échelle).